<a href="https://colab.research.google.com/github/Sineth7077/project-aura/blob/main/notebooks/02_data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Aura 2.0 - Notebook 02: Data IngestionThis notebook handles data ingestion for computer vision workflows:- Load sample image datasets- Preprocess images using `src.vision.preprocessing`- Upload/download from Google Cloud Storage (optional)- Save processed data for trainingImports reusable modules from `src/` where practical.

In [ ]:
import osimport sysimport numpy as npfrom pathlib import PathPROJECT_DIR = "/content/project-aura"if PROJECT_DIR not in sys.path:    sys.path.insert(0, PROJECT_DIR)!pip install --quiet pillowfrom src.core.config import load_configfrom src.vision.preprocessing import resize_image, normalize_image, prepare_batchfrom src.utils.gcs import GCSClientfrom src.utils.logging import get_loggerlogger = get_logger("notebook-02")config = load_config()print("Configuration loaded")

In [ ]:
DATA_DIR = "/content/project-aura/data"RAW_DIR = os.path.join(DATA_DIR, "raw")PROCESSED_DIR = os.path.join(DATA_DIR, "processed")os.makedirs(RAW_DIR, exist_ok=True)os.makedirs(PROCESSED_DIR, exist_ok=True)print("Data directories created:")print("  Raw:", RAW_DIR)print("  Processed:", PROCESSED_DIR)

In [ ]:
from PIL import Imagedef create_synthetic_image(path, size=(200, 200), color=None):    """Create a synthetic image file."""    if color is None:        color = (            np.random.randint(0, 255),            np.random.randint(0, 255),            np.random.randint(0, 255),        )    img = Image.new("RGB", size, color)    img.save(path)    return pathNUM_SAMPLES = 20sample_paths = []for i in range(NUM_SAMPLES):    path = os.path.join(RAW_DIR, f"sample_{i:03d}.jpg")    create_synthetic_image(path)    sample_paths.append(path)print(f"Created {len(sample_paths)} synthetic images in {RAW_DIR}")

In [ ]:
from PIL import Imageprocessed_paths = []for img_path in sample_paths:    img = Image.open(img_path).convert("RGB")    img_array = np.array(img)        resized = resize_image(        img_array,        target_size=config.vision.input_size,        preserve_aspect_ratio=True,    )        normalized = normalize_image(resized)        from src.vision.preprocessing import denormalize_image    denormalized = denormalize_image(normalized)        output_path = os.path.join(        PROCESSED_DIR,        os.path.basename(img_path),    )    Image.fromarray(denormalized).save(output_path)    processed_paths.append(output_path)print(f"Preprocessed {len(processed_paths)} images")print("Sample processed shape:", np.array(Image.open(processed_paths[0])).shape)

In [ ]:
images = []for p in processed_paths[:5]:    img = np.array(Image.open(p).convert("RGB"))    images.append(img)batch = prepare_batch(images, target_size=config.vision.input_size)print("Batch shape:", batch.shape)print("Batch dtype:", batch.dtype)print("Ready for model input")

In [ ]:
# Configure GCS clientgcs = GCSClient(    bucket_name=os.getenv("GCS_BUCKET_NAME"),    project_id=os.getenv("GCP_PROJECT_ID"),)# Test connectionif gcs._ensure_client():    print("GCS client connected successfully")    blobs = gcs.list_blobs(prefix="", bucket_name=os.getenv("GCS_BUCKET_NAME"))    print(f"Found {len(blobs)} blobs in bucket")else:    print("GCS client not available")    print("This is expected in minimal Colab environments")

In [ ]:
print("\n" + "="*60)print("Notebook 02 Complete - Data Ingestion")print("="*60)print(f"Raw images: {len(sample_paths)}")print(f"Processed images: {len(processed_paths)}")print(f"Batch tensor shape: {batch.shape}")print(f"Data directory: {DATA_DIR}")print("\nNext: Run 03_model_training.ipynb to train a model")